# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today we'll rewrite the products into a standard format.  
LLMs are great at this!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Re-writing</h2>
            <span style="color:#181;">LLMs have made it simple to do something that was considered impossible only a few years ago.
            This approach can be applied to almost any business vertical, and it's similar to the advanced techniques
            we used on Week 5.</span>
        </td>
    </tr>
</table>

In [4]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

True

# The next cell is where you choose Dataset

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

## For this lab

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30

In [5]:
LITE_MODE = True

In [6]:
username = "khirodsahoo93"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

README.md:   0%|          | 0.00/738 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/23.5M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.02M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.02M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 22,000 items
title='Odyssey K45120BLK Krom Utility/Record Case, Black' category='Musical_Instruments' price=119.95 full='Odyssey  Krom Utility/Record Case, Black\n[\'Protect your rare and hard to find records used for your special 45 rpm vinyl sets or just for safe keeping in our Krom™ series  black utility/record cases. Each has two individual compartments that hold a total of 120 7" vinyl records, 60 in each section, and can also be used as a utility case for cables and other essentials. Features include chrome plated hardware, a removable lid, a fully foam-lined interior and a heavy-duty latch and spring-loaded handle. Individual K45120 cases can be stacked on each other for convenient storage. Also available in silver ||.\']\n[\'Butterfly latch and spring-loaded handle\', \'Padlock latch holes\', \'Holds 120 7" vinyl records, 60 per compartment\', \'Foam-lined interior\', \'Rubber feet and stacking lid\']\n{"Item Weight": "7 pounds", "Product Dimensions": "9.25 x 17.75 x 9.7

In [9]:
train[100]

<Rolife DIY Book Nook Kit 3D Wooden Puzzle, Bookshelf Indert Decor with LED DIY Bookend Diorama Dollhouse Kit Crafts Hobbies Gifts for Adults/Teens (Sunshine Town) = $46.99>

In [7]:
items[2].id

In [10]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [11]:


SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [12]:
print(items[0].full)

Odyssey  Krom Utility/Record Case, Black
['Protect your rare and hard to find records used for your special 45 rpm vinyl sets or just for safe keeping in our Krom™ series  black utility/record cases. Each has two individual compartments that hold a total of 120 7" vinyl records, 60 in each section, and can also be used as a utility case for cables and other essentials. Features include chrome plated hardware, a removable lid, a fully foam-lined interior and a heavy-duty latch and spring-loaded handle. Individual K45120 cases can be stacked on each other for convenient storage. Also available in silver ||.']
['Butterfly latch and spring-loaded handle', 'Padlock latch holes', 'Holds 120 7" vinyl records, 60 per compartment', 'Foam-lined interior', 'Rubber feet and stacking lid']
{"Item Weight": "7 pounds", "Product Dimensions": "9.25 x 17.75 x 9.75 inches", "Is Discontinued By Manufacturer": "No", "Date First Available": "September 6, 2012", "Color Name": "BLACK", "Material Type": "Rubbe

In [13]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


Title: Odyssey Krom Utility/Record Case, Black  
Category: Music Accessories  
Brand: Odyssey  
Description: Protective case for 120 7" vinyl records with dual compartments, stackable, and multifunctional for cables.  
Details: Chrome-plated hardware, removable lid, foam-lined interior, rubber feet, and toggle closure.

Input tokens: 416
Output tokens: 82
Cost: 0.006 cents


/opt/anaconda3/envs/llms/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 5: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='Title: O...d concise description.'), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st... concise description.')), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


In [ ]:

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


In [14]:
MODEL = "openai/gpt-oss-20b"


In [15]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [26]:
items[0]

<Odyssey K45120BLK Krom Utility/Record Case, Black = $119.95>

In [25]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "openai/gpt-oss-20b", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Odyssey  Krom Utility/Record Case, Black\\n[\'Protect your rare and hard to find records used for your special 45 rpm vinyl sets or just for safe keeping in our Krom\\u2122 series  black utility/record cases. Each has two individual compartments that hold a total of 120 7\\" vinyl records, 60 in each section, and can also be used as a utility case for cables and other essentials. Features include chrome plated hardware, a removable lid, a fully foam-lined interior and a heavy-duty latch and spring-loaded handle. Individual K45120 cases can be stacked

In [27]:

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [28]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [29]:
import os
from groq import Groq

groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [30]:

with open("jsonl/0_1000.jsonl", "rb") as f:
    response = groq.files.create(file=f, purpose="batch")
response

FileCreateResponse(id='file_01kfcfsn4pe08a85n9ks4gkg47', bytes=2517397, created_at=1768872006, filename='0_1000.jsonl', object='file', purpose='batch', size=0, md5='pgZFSb5/WoUouO7o59mnww==', content_type='application/jsonl')

In [31]:
file_id = response.id
file_id

'file_01kfcfsn4pe08a85n9ks4gkg47'

In [32]:
response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response

BatchCreateResponse(id='batch_01kfcfstg2f15v9cbm1kjfrhxm', completion_window='24h', created_at=1768872012, endpoint='/v1/chat/completions', input_file_id='file_01kfcfsn4pe08a85n9ks4gkg47', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1768958412, failed_at=None, finalizing_at=None, in_progress_at=None, metadata=None, output_file_id=None, request_counts=RequestCounts(completed=0, failed=0, total=0), project_id='project_01kfbqee3tecst9beq0c5p2rwf')

In [37]:
result = groq.batches.retrieve(response.id)
result

BatchRetrieveResponse(id='batch_01kfcfstg2f15v9cbm1kjfrhxm', completion_window='24h', created_at=1768872012, endpoint='/v1/chat/completions', input_file_id='file_01kfcfsn4pe08a85n9ks4gkg47', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1768872029, error_file_id=None, errors=None, expired_at=None, expires_at=1768958412, failed_at=None, finalizing_at=1768872026, in_progress_at=1768872017, metadata=None, output_file_id='file_01kfcft92zenttz6bgtcv4vn4c', request_counts=RequestCounts(completed=1000, failed=0, total=1000), project_id='project_01kfbqee3tecst9beq0c5p2rwf')

In [38]:
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

In [39]:
with open("jsonl/batch_results.jsonl", "r") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary


In [41]:
print(items[0].summary)

Title: Odyssey Krom Utility/Record Case – Black  
Category: Storage & Organization  
Brand: Odyssey  
Description: A durable black utility case that safely holds up to 120 7" vinyl records in two compartments and doubles as a cable organizer.  
Details: Features chrome‑plated hardware, a removable lid, a foam‑lined interior, a heavy‑duty latch with padlock holes, a butterfly latch, spring‑loaded handle, and rubber feet for easy stacking.


In [42]:
print(items[1000].summary)

None


## I've put exactly this logic into a Batch class

- Divides items into groups of 1,000
- Kicks off batches for each
- Allows us to monitor and collect the results when complete

## COSTS

Using Groq, for me - this cost under $1 for the Lite dataset and under $30 for the big dataset

But you don't need to pay anything! In the next lab, you can load my pre-processed results

In [43]:
Batch.create(items, LITE_MODE)

Created 22 batches


In [44]:
Batch.run()

  0%|          | 0/22 [00:00<?, ?it/s]

Submitted 22 batches


In [45]:
Batch.fetch()

  0%|          | 0/22 [00:00<?, ?it/s]

Finished 22 of 22 batches


In [46]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [51]:
print(items[20000].summary)

Title: Musotica 2023 2-in-1 Cordless Air Duster & Vacuum  
Category: Electronics  
Brand: Musotica  
Description: A rechargeable, lightweight 2‑in‑1 air duster and handheld vacuum with LED lighting for versatile dusting and cleaning.  
Details: Features three speed modes, eight interchangeable nozzles/brushes, a long silicone tube, and a brushless motor for powerful, quiet operation.


In [52]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

## Push the final dataset to the hub

If lite mode, we'll only push the lite dataset

If full mode, we'll push both datasets (in case you decide to use lite later)

In [53]:
username = "khirodsahoo93"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/20 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


## And here they are!

https://huggingface.co/datasets/ed-donner/items_lite

https://huggingface.co/datasets/ed-donner/items_full
